# 11 - Proposal Writer Agent
## Multi-Agent Document Generation: Supervisor + Writer Pipeline
25 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/11-proposal-writer/proposal_writer_workbook.ipynb)

Writing a winning RFP response requires two very different jobs: analysing what the
client actually wants, then crafting a response that addresses those wants persuasively.

This example separates those jobs into two agents:
1. **Supervisor** -- reads the RFP, extracts requirements, identifies win themes
2. **Writer** -- receives the structured outline and drafts the full proposal

Neither agent tries to do both jobs. The supervisor's output is a typed contract
that constrains and focuses the writer.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | Why two agents beat one for document generation |
| 2 | Schema: RFPRequirement + ProposalOutline + Proposal |
| 3 | Stage 1: Supervisor decomposes the RFP |
| 4 | Stage 2: Writer drafts using the outline as context |
| 5 | Run the full pipeline on a real-world RFP sample |
| * | Exercises + Answer Key |

### Prerequisites
- Python 3.10+, or Google Colab
- OPENAI_API_KEY in .env or Colab Secrets

In [ ]:
try:
    from google.colab import userdata  # noqa: F401
    import subprocess
    subprocess.run(["pip", "install", "-q", "langchain-openai", "pydantic", "python-dotenv"], check=True)
    print("Packages installed.")
except ImportError:
    print("Local environment.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("Set OPENAI_API_KEY in .env or Colab Secrets.")
print("API key ready.")

---

## Part 1 - Why Two Agents?

A single LLM call that receives an RFP and outputs a full proposal tends to:
- Miss mandatory requirements buried in the small print
- Write generic sections that don't address the evaluation criteria
- Forget the client's stated priorities by the time it reaches section 4

The two-agent pattern solves this by separating concerns:

| Stage | Job | Output |
|-------|-----|--------|
| Supervisor | What does the client ACTUALLY want? | ProposalOutline |
| Writer | How do we win? | Proposal |

The supervisor output becomes a grounding contract. The writer receives:
- The raw RFP (for tone and context)
- The extracted win themes (must appear in every section)
- The mandatory requirements (must appear in compliance_statement)
- The evaluation criteria (determines where to invest words)

The writer cannot miss a mandatory requirement it has been handed explicitly.

---

## Part 2 - Schema

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field


class RFPRequirement(BaseModel):
    section: str = Field(description="Which section or question this requirement comes from")
    requirement: str = Field(description="The specific thing the client is asking for")
    mandatory: bool = Field(description="True if this is a pass/fail requirement")


class ProposalOutline(BaseModel):
    """Structured decomposition of an RFP produced by the supervisor."""

    client_name: Optional[str] = None
    rfp_title: str
    submission_deadline: Optional[str] = None
    requirements: List[RFPRequirement] = Field(
        description="All requirements extracted from the RFP, mandatory ones first"
    )
    win_themes: List[str] = Field(
        description="2-4 strategic themes that should run through the whole proposal"
    )
    evaluation_criteria: List[str] = Field(
        description="How the client will score proposals, in priority order"
    )
    sections_to_write: List[str] = Field(
        description="Ordered list of proposal sections that need to be drafted"
    )


class Proposal(BaseModel):
    """Final assembled proposal document."""

    client_name: Optional[str] = None
    rfp_title: str
    win_themes: List[str]
    executive_summary: str = Field(
        description="Why we win -- lead with the client's problem and our unique answer"
    )
    our_approach: str = Field(
        description="Methodology and how we would deliver the work"
    )
    team_and_credentials: str = Field(
        description="Relevant experience, key personnel, and why we are best placed to deliver"
    )
    timeline: str = Field(
        description="Phased project plan with key milestones and deliverables"
    )
    commercial: str = Field(
        description="Pricing structure, value drivers, and commercial terms"
    )
    why_us: str = Field(
        description="Differentiators -- what we offer that competitors cannot match"
    )
    key_differentiators: List[str] = Field(
        description="3-5 bullet-point differentiators for the cover page or summary slide"
    )
    compliance_statement: str = Field(
        description="Confirmation that all mandatory requirements are met"
    )

print("Schema defined.")
print("Handoff: Supervisor -> ProposalOutline -> Writer -> Proposal")

---

## Part 3 - Stage 1: Supervisor Decomposes the RFP

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

SUPERVISOR_SYSTEM = SystemMessage(
    """You are a proposal director reviewing an RFP before your team drafts a response.

Your job is to produce a structured decomposition of the RFP:
1. Extract every requirement, marking which are mandatory pass/fail criteria
2. Identify 2-4 win themes -- the strategic angles that should run through the whole proposal
3. Identify how the client will evaluate and score proposals
4. List the sections the proposal should contain, in order

Be precise. Requirements you miss now become compliance failures later."""
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
supervisor = SUPERVISOR_SYSTEM | llm.with_structured_output(ProposalOutline)

SAMPLE_RFP = """
REQUEST FOR PROPOSAL
Digital Transformation Office Maturity Assessment and Roadmap
Issued by: Meridian Financial Services Group
RFP Reference: MFSG-2025-DT-011
Submission Deadline: 30 March 2025

1. BACKGROUND
Meridian Financial Services Group (MFSG) is a mid-market financial services firm with
1,400 employees across retail banking, wealth management, and insurance divisions.
Our Digital Transformation Office (DTO) was established in 2022 but has struggled
to demonstrate measurable ROI. Executive leadership requires an independent assessment
of the DTO's current maturity, operating model, and a prioritised 18-month roadmap
before budget allocation for FY2026 is finalised.

2. SCOPE OF WORK
a) Maturity Assessment: Benchmark the DTO against industry frameworks across five
   dimensions: strategy, governance, talent, technology, and delivery.
b) Stakeholder Interviews: Minimum 20 stakeholders including C-suite sponsors,
   DTO leadership, and business unit heads.
c) Root Cause Analysis: Primary drivers of underperformance with supporting evidence.
d) Prioritised Roadmap: 18-month roadmap with effort vs. impact ranking, quick wins
   (0-90 days) clearly identified.
e) Executive Presentation: Findings to MFSG Executive Committee within 10 weeks.

3. MANDATORY REQUIREMENTS (PASS/FAIL)
M1. Minimum three digital maturity assessments for regulated financial services
    clients in the past five years.
M2. Lead partner: minimum 15 years experience in financial services transformation.
M3. All data handling compliant with UK GDPR and FCA data governance requirements.
M4. Fixed-fee commercial structure required; time-and-materials bids disqualified.
M5. Draft deliverables within six weeks of contract signature.

4. EVALUATION CRITERIA (scored out of 100)
- Relevant sector experience and case studies: 35 points
- Quality and specificity of proposed methodology: 30 points
- Team credentials and named personnel: 20 points
- Commercial value and fixed-fee clarity: 15 points

5. COMMERCIAL
Single fixed fee. Budget envelope: GBP 180,000 exclusive of VAT.
"""

outline = supervisor.invoke(HumanMessage(content="RFP to decompose:\n\n" + SAMPLE_RFP))

mandatory = [r for r in outline.requirements if r.mandatory]
optional = [r for r in outline.requirements if not r.mandatory]

print(f"RFP: {outline.rfp_title}")
print(f"Client: {outline.client_name}")
print(f"Deadline: {outline.submission_deadline}")
print(f"\nRequirements: {len(mandatory)} mandatory, {len(optional)} optional")
print("\nMandatory (pass/fail):")
for r in mandatory:
    print(f"  [{r.section}] {r.requirement}")
print(f"\nWin themes ({len(outline.win_themes)}):")
for t in outline.win_themes:
    print(f"  * {t}")
print("\nEvaluation criteria:")
for i, c in enumerate(outline.evaluation_criteria, 1):
    print(f"  {i}. {c}")

---

## Part 4 - Stage 2: Writer Drafts Using the Outline

In [ ]:
WRITER_SYSTEM = SystemMessage(
    """You are a senior proposal writer crafting a winning RFP response.

You will receive:
- The original RFP text
- A structured outline (win themes, requirements, evaluation criteria)

Your draft must:
- Lead every section with the client's problem, not your firm's capabilities
- Weave the win themes consistently through all sections
- Be specific about methodology -- no generic consulting language
- Address all mandatory requirements explicitly in the compliance_statement
- Keep the executive_summary under 200 words and punchy

Write to win, not to comply."""
)

writer = WRITER_SYSTEM | llm.with_structured_output(Proposal)

context = (
    "RFP text:\n" + SAMPLE_RFP + "\n\n"
    "Win themes: " + ", ".join(outline.win_themes) + "\n"
    "Evaluation criteria: " + ", ".join(outline.evaluation_criteria) + "\n"
    "Mandatory requirements:\n"
    + "\n".join(
        "- [" + r.section + "] " + r.requirement
        for r in outline.requirements
        if r.mandatory
    )
)

proposal = writer.invoke(context)

print(f"Proposal drafted for: {proposal.client_name}")
print(f"Win themes threaded: {', '.join(proposal.win_themes)}")
print("\n--- EXECUTIVE SUMMARY ---")
print(proposal.executive_summary)
print("\nKey differentiators:")
for d in proposal.key_differentiators:
    print(f"  * {d}")

---

## Part 5 - Full Pipeline Output

In [ ]:
print("=" * 60)
print("FULL PROPOSAL")
print("=" * 60)

print("\n--- OUR APPROACH ---")
print(proposal.our_approach)

print("\n--- TEAM & CREDENTIALS ---")
print(proposal.team_and_credentials)

print("\n--- TIMELINE ---")
print(proposal.timeline)

print("\n--- COMMERCIAL ---")
print(proposal.commercial)

print("\n--- WHY US ---")
print(proposal.why_us)

print("\n--- COMPLIANCE STATEMENT ---")
print(proposal.compliance_statement)

---

## Exercise 1 - Add a Scoring Matrix

The evaluation criteria have weights (35/30/20/15 points). Add a `ScoringAssessment`
model that estimates how well the proposal scores against each criterion, and
extend `Proposal` to include it.

```python
class CriterionScore(BaseModel):
    criterion: str
    max_points: int
    estimated_score: int
    rationale: str

class ScoringAssessment(BaseModel):
    criteria_scores: List[CriterionScore]
    total_estimated: int
    total_possible: int
    confidence: Literal["high", "medium", "low"]
```

Add a `scoring_assessment` field to `Proposal` and update the writer system prompt
to populate it.

In [ ]:
# Exercise 1 Starter
class CriterionScore(BaseModel):
    criterion: str
    max_points: int
    estimated_score: int
    rationale: str

# TODO: Create ScoringAssessment with criteria_scores, total_estimated, total_possible, confidence
# TODO: Extend Proposal to include scoring_assessment: Optional[ScoringAssessment]
# TODO: Update WRITER_SYSTEM to instruct the model to self-assess against each criterion
# TODO: Run and print the scoring table

In [ ]:
# Exercise 1 Answer Key
class ScoringAssessment(BaseModel):
    criteria_scores: List[CriterionScore]
    total_estimated: int
    total_possible: int
    confidence: Literal["high", "medium", "low"]


class ScoredProposal(BaseModel):
    client_name: Optional[str] = None
    rfp_title: str
    win_themes: List[str]
    executive_summary: str
    our_approach: str
    team_and_credentials: str
    timeline: str
    commercial: str
    why_us: str
    key_differentiators: List[str]
    compliance_statement: str
    scoring_assessment: Optional[ScoringAssessment] = None


SCORED_WRITER_SYSTEM = SystemMessage(
    """You are a senior proposal writer crafting a winning RFP response.

Draft all proposal sections as instructed. Additionally, populate scoring_assessment
by self-evaluating the draft against each evaluation criterion:
- Be honest: a mediocre credential section should not score 19/20
- Include rationale for each score
- Sum scores into total_estimated and set confidence based on how well
  the draft evidences claims

Write to win, not to comply."""
)

scored_writer = SCORED_WRITER_SYSTEM | llm.with_structured_output(ScoredProposal)

scored_context = (
    "RFP text:\n" + SAMPLE_RFP + "\n\n"
    "Win themes: " + ", ".join(outline.win_themes) + "\n"
    "Evaluation criteria with weights:\n"
    "- Relevant sector experience and case studies: 35 points\n"
    "- Quality and specificity of proposed methodology: 30 points\n"
    "- Team credentials and named personnel: 20 points\n"
    "- Commercial value and fixed-fee clarity: 15 points\n"
    "Mandatory requirements:\n"
    + "\n".join(
        "- [" + r.section + "] " + r.requirement
        for r in outline.requirements
        if r.mandatory
    )
)

scored_proposal = scored_writer.invoke(scored_context)

if scored_proposal.scoring_assessment:
    sa = scored_proposal.scoring_assessment
    print(f"SCORING ASSESSMENT (confidence: {sa.confidence})")
    print(f"{'Criterion':<45} {'Score':>6} {'Max':>4}")
    print("-" * 58)
    for cs in sa.criteria_scores:
        print(f"{cs.criterion[:44]:<45} {cs.estimated_score:>6} {cs.max_points:>4}")
        print(f"  -> {cs.rationale}")
    print("-" * 58)
    print(f"{'TOTAL':<45} {sa.total_estimated:>6} {sa.total_possible:>4}")

---

## What You Built

A two-agent proposal pipeline demonstrating multi-agent document generation:

1. **Supervisor** extracts a typed `ProposalOutline` from the RFP -- requirements
   flagged mandatory, win themes identified, evaluation criteria ranked
2. **Writer** receives the outline as a grounding contract alongside the raw RFP --
   cannot miss mandatory requirements or stray from win themes
3. The `compliance_statement` field in `Proposal` acts as a forced completeness check

The pattern scales: add a third Reviewer agent to critique the draft, or a
Pricing agent that populates the commercial section from a rate card.

---
*Example 11 of 23 - agent-use-cases*